# Pasang head konteks ke SongMap Debug

Buka dari root **SongMap** di Mac. Resource worship palette6 yang sebelumnya dipakai aplikasi harus sudah terpasang. Notebook ini menguji sumber Swift/matematikanya, memasang **hanya model tambahan** yang lulus paritas, lalu membuka proyek Xcode.

Model baru berada di `SongMap/Resources/MLModels/SongFormerContext`. Resource lama `SongFormerWorship` tetap dipertahankan. Installer memeriksa bahwa keduanya berasal dari validation palette6 yang sama. File model hasil instalasi diabaikan oleh Git.

Tidak ada merge, upload TestFlight atau perubahan jalur Release dari notebook ini. Inferensi aplikasi memakai CPU_ONLY. Build penuh iOS dan hasil iPhone masih perlu diuji.


In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, json, os

ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/install_section_context.py').is_file(), 'Buka notebook dari root SongMap'
# Jangan resolve symlink interpreter: Python harus tetap dijalankan dari folder venv.
RUN_PYTHON=ROOT.parent/'Section-Annotations/.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
subprocess.run([str(RUN_PYTHON), '-u', '-c',
                'import sys; print("Python:", sys.executable); print("Environment:", sys.prefix); import numpy, torch; print("NumPy/PyTorch OK")'],
               cwd=ROOT, check=True)
reports=list((ROOT/'outputs.local').glob('context-coreml-fp32-*/parity/validation-all.json'))
assert reports, 'Selesaikan section_context_coreml_fp32.ipynb dahulu'
PARITY_REPORT=max(reports,key=lambda p:p.stat().st_mtime).resolve()
proof=json.loads(PARITY_REPORT.read_text())
assert proof['passed'] and proof['full_validation'] and proof['section_agreement']==1
log=ROOT/'outputs.local'/('app-context-install-'+datetime.now().strftime('%Y%m%d-%H%M%S')+'.log')
print('Paket yang telah diperiksa:',PARITY_REPORT)
print('Log:',log)

def run_logged(command,env=None):
    log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Gagal ({status}); lihat {log}. Jangan lanjut build sebelum langkah ini selesai.')


In [ ]:
# Membandingkan pooling/timing/koreksi Swift dengan referensi Python sintetis.
# Juga mengompilasi sumber runtime konteks sebenarnya menggunakan SDK Mac.
run_logged(['bash',str(ROOT/'scripts/test_section_context.sh')],
           env={**os.environ,'SECTION_CONTEXT_PYTHON':str(RUN_PYTHON)})


In [ ]:
run_logged([str(RUN_PYTHON),str(ROOT/'scripts/install_section_context.py'),
            '--parity-report',str(PARITY_REPORT)])
manifest=ROOT/'SongMap/Resources/MLModels/SongFormerContext/songformer_section_context_manifest.json'
receipt=json.loads(manifest.read_text())
print('Terpasang:',manifest.parent)
print('Model:',receipt['model'],'| epoch:',receipt['selectedEpoch'],'| precision:',receipt['precision'])


In [ ]:
# Pertahankan proyek dan pengaturan signing lokal yang sudah ada.
project=ROOT/'SongMap.xcodeproj'
if not (project/'project.pbxproj').is_file():
    run_logged([str(RUN_PYTHON),str(ROOT/'scripts/generate_xcode_project.py')])
subprocess.run(['open',str(project)],check=True)


Di Xcode:

1. Pilih target iPhone 17 dan konfigurasi **Debug**.
2. **Product → Scheme → Edit Scheme → Run → Arguments → Arguments Passed On Launch**.
3. Nonaktifkan `--songformer-legacy` dan `--songformer-worship-no-context` jika ada. Tambahkan **`--songformer-worship-context`**.
4. Build dan jalankan, lalu analisis ulang satu lagu. Pilihan ini diingat untuk pembukaan aplikasi berikutnya dari Home Screen.

Periksa log **`WORSHIP ANALYSIS_START v10 ... section_context=epoch5-fp32`** dan **`WORSHIP CONTEXT_DONE`**. Setelah selesai, Library → menu titik tiga → **Analysis reports → Share report**. Kirim JSON baru tersebut.

Laporan memuat `contextEvidence`: label dan probabilitas sebelum/sesudah koreksi, coverage, serta hash input/model. Frame dalam `structureEvidence` tetap merupakan prediksi dasar SongFormer. Fitur audio/embedding tidak diekspor dalam laporan. Waktu pooling, model load, inferensi dan postprocessing tercatat terpisah sebagai metric anak SongFormer.

Untuk membandingkan baseline, ganti argumen context dengan **`--songformer-worship-no-context`**; jangan membiarkan keduanya aktif. Batas section tetap berasal dari MusicUnderstanding, sehingga koreksi label tidak menjamin batas outro/bridge bergeser. Analisis ulang diperlukan; hasil analisis lama tidak ditulis ulang otomatis.
